1️⃣ CLI 강조 함수
처리 규칙 (간단 버전):
* -또는 --로 시작 → 빨강
* ' 또는 " 문자열 전체 → 파랑
* 숫자 → 주황
* %(key)s 형태
    * %( 와 )s → 하늘색
    * key → 초록

In [2]:
import html
import shlex
import re
import json
from pprint import pformat
from pathlib import Path
import webbrowser


# 색상 정의
COLORS = {
    "option": "#d73a49",      # 빨강
    "string": "#005cc5",      # 파랑
    "number": "#e36209",      # 주황
    "format_wrap": "#1abcdd", # 하늘
    "format_key": "#22863a",  # 초록
    "key": "#6f42c1",         # dict key 보라
    "bool": "#b31d28",
    "null": "#6a737d",
}


def span(text, color):
    return f'<span style="color:{color}">{text}</span>'


def highlight_cli(cli_string: str) -> str:
    tokens = shlex.split(cli_string, posix=False)
    highlighted = []

    format_pattern = re.compile(r'%\(([^)]+)\)([a-zA-Z])')

    for token in tokens:
        escaped = html.escape(token)

        # 옵션
        if token.startswith("-"):
            highlighted.append(span(escaped, COLORS["option"]))
            continue

        # 숫자
        if token.isdigit():
            highlighted.append(span(escaped, COLORS["number"]))
            continue

        # 문자열 (따옴표 포함)
        if (token.startswith('"') and token.endswith('"')) or \
           (token.startswith("'") and token.endswith("'")):

            quote = token[0]
            inner = token[1:-1]

            parts = []
            last = 0

            for m in format_pattern.finditer(inner):
                start, end = m.span()
                key = m.group(1)
                typ = m.group(2)

                # 일반 문자열 부분
                normal = html.escape(inner[last:start])
                parts.append(normal)

                # %( 와 )s 분리 강조
                parts.append(span("%(", COLORS["format_wrap"]))
                parts.append(span(html.escape(key), COLORS["format_key"]))
                parts.append(span(f"){typ}", COLORS["format_wrap"]))

                last = end

            parts.append(html.escape(inner[last:]))

            result = (
                span(html.escape(quote), COLORS["string"])
                + "".join(parts)
                + span(html.escape(quote), COLORS["string"])
            )

            highlighted.append(result)
            continue

        highlighted.append(escaped)

    return "<div style='font-family:monospace'>" + " ".join(highlighted) + "</div>"

2️⃣ dict 강조 함수 (정렬 + pprint 스타일)
요구사항:
- dict 객체 직접 입력
- 키 정렬
- JSON처럼 정렬된 구조 출력
- 타입별 색상 강조

In [3]:
def highlight_dict(data: dict) -> str:
    sorted_dict = dict(sorted(data.items(), key=lambda x: x[0]))
    formatted = json.dumps(sorted_dict, indent=4, ensure_ascii=False)

    lines = formatted.split("\n")
    highlighted_lines = []

    for line in lines:
        escaped = html.escape(line)

        # key 강조
        escaped = re.sub(
            r'"([^"]+)":',
            lambda m: span(f'"{m.group(1)}"', COLORS["key"]) + ":",
            escaped
        )

        # 숫자 강조
        escaped = re.sub(
            r'\b\d+\b',
            lambda m: span(m.group(0), COLORS["number"]),
            escaped
        )

        # true / false
        escaped = re.sub(
            r'\b(true|false)\b',
            lambda m: span(m.group(0), COLORS["bool"]),
            escaped
        )

        # null
        escaped = re.sub(
            r'\bnull\b',
            lambda m: span(m.group(0), COLORS["null"]),
            escaped
        )

        highlighted_lines.append(escaped)

    return "<div style='font-family:monospace'>" + "<br>".join(highlighted_lines) + "</div>"

3️⃣ HTML 렌더링 함수
브라우저에서 바로 열도록 구현.

In [4]:
def render_html(html_string: str, filename="preview.html"):
    full_html = f"""
    <html>
    <head>
        <meta charset="utf-8">
    </head>
    <body style="background:#f6f8fa;">
        {html_string}
    </body>
    </html>
    """

    path = Path(filename)
    path.write_text(full_html, encoding="utf-8")
    webbrowser.open(path.resolve().as_uri())

# 예제


In [5]:

if __name__ == "__main__":
    cli = 'yt-dlp -f "bv+ba" --output \'%(title)s.%(ext)s\' -N 3 URL'
    html_cli = highlight_cli(cli)
    render_html(html_cli, "cli.html")

    data = {
        "format": "bv+ba",
        "retries": 3,
        "output": "%(title)s.%(ext)s",
        "writeinfojson": True,
        "metadata": None
    }

    html_dict = highlight_dict(data)
    render_html(html_dict, "dict.html")